# [LAB] LangGraph로 그래프 Agent 만들기

완성된 코드로 LangGraph 그래프를 다섯 단계로 실행해 봅니다. `# ✍️ CHANGE_HERE:` 자리에서는 값을 바꿔 다시 실행하고, 동작이 어떻게 달라지는지 봅니다.

- **2장 기본 그래프**: State, 노드 하나, START → chatbot → END를 잇고 컴파일합니다.
- **3장 조건부 엣지**: 질문의 언어를 감지해 한국어·영어 답변 노드로 나눠 보냅니다.
- **4장 ToolNode로 ReAct 루프**: 도구 호출 → 도구 실행 → 다시 판단하는 반복을 그래프로 만듭니다.
- **5장 사람 승인**: 도구 실행 직전에 멈추고, 승인하면 이어 가고 거부하면 막습니다.
- **6장 대화형 승인**: 사람이 입력창에서 직접 승인·거부를 고릅니다.

셀은 위에서부터 차례로 실행합니다. 각 장 첫머리의 접힌 준비 코드도 실행해야 그 아래 셀이 동작합니다.

**목차**

- 1. 실습 방법
- 2. 기본 그래프 만들기
  - 2.1 State와 chatbot 노드 만들기
  - 2.2 노드와 엣지 잇기
  - 2.3 컴파일하고 실행하기
- 3. 조건부 엣지로 언어별 분기하기
  - 3.1 State와 노드 정의
  - 3.2 라우팅 함수
  - 3.3 그래프 조립
  - 3.4 질문 넣어 실행하기
- 4. ToolNode로 ReAct 루프 만들기
  - 4.1 도구 정의
  - 4.2 도구 목록과 LLM 연결
  - 4.3 agent 노드
  - 4.4 ToolNode로 루프 잇고 실행하기
  - 4.5 create_agent로 한 번에 만들기
- 5. 사람 승인 넣기
  - 5.1 도구와 그래프 조립
  - 5.2 체크포인터와 중단 지점
  - 5.3 1단계: 도구 실행 직전에 멈추기
  - 5.4 2단계: 승인하고 이어 가기
  - 5.5 거부하고 이어 가기
- 6. 대화형 승인 Agent
  - 6.1 도구와 그래프 조립
  - 6.2 승인·거부를 묻는 함수
  - 6.3 Agent 실행 루프
  - 6.4 직접 실행하기
- 7. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env()

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고칩니다. `.env`는 노트북을 시작할 때 한 번만 읽으므로, 고친 뒤에는 커널을 다시 시작하고 맨 위 셀부터 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

도구를 부를 때 `Error in ConsoleCallbackHandler.on_tool_start callback: KeyError('input')` 같은 줄이 섞여 보일 수 있습니다. 디버그 내용을 출력하는 쪽의 문제라 실행 결과에는 영향이 없으니 그대로 두면 됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

## 1. 실습 방법

코드는 모두 완성되어 있습니다. 위에서부터 차례로 실행합니다.

- `# ✍️ CHANGE_HERE:` 주석이 있는 셀은 먼저 그대로 실행해 결과를 봅니다. 그다음 셀 위의 **바꿔 볼 값**을 복사해, 주석 바로 아래 줄의 표시된 자리(지금 값)를 지우고 붙여 넣습니다. 어떻게 달라질지 생각해 본 뒤 다시 실행합니다.
- 달라진 결과를 본 뒤 셀 아래 접힌 **무슨 일이 생기나**를 펼쳐 비교합니다. 확인했으면 값을 원래대로 되돌립니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. 기본 그래프 만들기

노드 하나짜리 가장 작은 그래프를 만듭니다. State(공유 데이터)와 Node(작업 단위)를 정하고, Edge(다음 이동)로 START → chatbot → END를 이은 뒤, 컴파일해서 질문 하나를 실행합니다.

준비 코드입니다. 그래프를 만드는 도구를 불러오고 `llm = get_llm()`으로 모델을 준비합니다. 접어 두었으니 실행만 합니다.

In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

### 2.1 State와 chatbot 노드 만들기

**✍️ CHANGE_HERE: `Annotated[list, add_messages]`를 `list`로 바꾸면 2.3의 메시지 목록은 어떻게 될까요?**

**바꿔 볼 값** — 바로 아래 줄의 `Annotated[list, add_messages]` 자리에 복사해 붙여 넣기

```text
list
```

이곳만 값이 아니라 Reducer(`add_messages`, 메시지를 합치는 규칙)를 빼 봅니다. 값을 바꿔서는 이 규칙이 하는 일이 보이지 않기 때문입니다. 이 셀부터 2.3까지 차례로 다시 실행하고, 확인했으면 되돌린 뒤 이 셀부터 2.3까지 다시 실행합니다.

노드는 State를 받아서, 바꿀 값만 담은 딕셔너리를 돌려주는 함수입니다. `chatbot_node`는 지금까지의 메시지 목록을 LLM에 넘기고, LLM의 답 메시지를 `messages`에 담아 돌려줍니다. 답이 하나여도 `[response]`처럼 대괄호로 감싼 목록으로 담습니다. `add_messages` 규칙 덕분에 돌려준 메시지는 기존 목록 뒤에 쌓입니다.

In [ ]:
# ── LangGraph 3대 요소 ──────────────────────────────────────────────────────

# 1. State: 그래프 전체에서 공유되는 데이터 구조.
#    TypedDict 또는 Pydantic BaseModel로 정의한다.
#    add_messages Reducer 덕분에 messages는 덮어쓰기가 아닌 누적(append)이다.


class State(TypedDict):
    # ✍️ CHANGE_HERE: `Annotated[list, add_messages]`를 `list`로 바꾸면 2.3의 메시지 목록은 어떻게 될까요?
    messages: Annotated[list, add_messages]


# 2. Node: State를 받아 State의 일부(dict)를 반환하는 함수.
#    반환된 dict는 Reducer 규칙에 따라 기존 State에 병합된다.
def chatbot_node(state: State) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

<details><summary>무슨 일이 생기나</summary>

2.3의 출력에서 `Human Message` 칸이 사라지고 `Ai Message` 칸 하나만 남습니다. 오류는 나지 않습니다.

`add_messages`가 없으면 노드가 돌려준 `[response]`가 기존 `messages` 목록 뒤에 쌓이지 않고, 목록을 통째로 바꿉니다. 그래서 질문이 State에서 지워집니다. 대화를 이어 가는 그래프가 앞의 메시지를 기억하려면 이 규칙이 있어야 합니다.

</details>

### 2.2 노드와 엣지 잇기

`builder`에 노드를 올리고 엣지로 잇습니다. `add_node`에는 노드 이름과 그 노드가 할 일(함수)을 주고, `add_edge`에는 출발 노드와 도착 노드를 줍니다. `START`와 `END`는 LangGraph가 주는 시작점과 끝점입니다.

In [ ]:
# 3. Edge: 노드 간 흐름을 정의한다.
#    START/END는 LangGraph가 제공하는 진입·종료 지점이다.
builder = StateGraph(State)
builder.add_node("chatbot", chatbot_node)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END);

### 2.3 컴파일하고 실행하기

`compile()`은 조립한 그래프를 실행할 수 있는 `app`으로 만듭니다. 이 셀은 컴파일한 뒤 그래프 구조를 그림으로 보여 주고, 질문 하나를 넣어 실행합니다.

In [ ]:
app = builder.compile()


# 그래프 구조를 그림으로 출력
from draw import draw

draw(app)
print()

basic_graph_result = app.invoke(
    {"messages": [HumanMessage(content="LangGraph가 뭔지 두 문장으로 설명해줘.")]}
)
results = basic_graph_result["messages"]
for m in results:
    m.pretty_print()

**볼 지점**
- 그래프 그림의 `__start__`와 `__end__`는 LangGraph가 붙이는 시작점과 끝점입니다. 그 사이에 추가한 `chatbot` 노드가 있습니다.
- 실행 결과의 `messages`에는 질문과 답이 차례로 쌓여 있습니다. 코드는 이 목록을 `results`로 꺼내 `pretty_print()`로 하나씩 보여 줍니다. `Human Message`가 질문, `Ai Message`가 LLM의 답입니다.

## 3. 조건부 엣지로 언어별 분기하기

질문의 언어를 먼저 알아낸 뒤, 한국어면 한국어 답변 노드로, 아니면 영어 답변 노드로 보냅니다. 갈 곳을 함수의 결과로 정하는 조건부 엣지를 씁니다.

마지막 실행 셀에서는 `# ✍️ CHANGE_HERE:` 줄의 질문을 바꿔 다시 실행해 봅니다.

이 장의 준비 코드입니다. 앞 장과 같은 도구에 `Literal`(정해진 값만 허용하는 타입 표시)을 더해 불러옵니다. 내용은 몰라도 되니 실행만 하고 넘어갑니다.

In [ ]:
from typing import Annotated, Literal

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

### 3.1 State와 노드 정의

`detect_language` 노드는 마지막 메시지의 언어를 LLM에게 물어 `"ko"`나 `"en"`으로 정하고, State의 `language` 필드에 기록합니다. 답 메시지가 아니라 `language` 값만 돌려주는 노드입니다. LLM의 답에는 다른 글자가 섞일 수 있으니, 답을 `"ko"`와 `"en"` 둘 중 하나로 정리해야 다음 라우팅 함수가 제대로 고릅니다. 정리한 값은 `lang`이라는 이름에 담습니다. 바로 아래 `return` 줄이 이 이름을 씁니다.

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    language: str  # 감지된 언어 ('ko' | 'en')


# ── 노드 정의 ────────────────────────────────────────────────────────────────


def detect_language(state: State) -> dict:
    """마지막 메시지의 언어를 감지해 State에 기록한다."""
    last = state["messages"][-1].content
    result = llm.invoke(
        f'다음 텍스트의 언어를 "ko" 또는 "en" 중 하나로만 판별해: {last}'
    )
    lang = "ko" if "ko" in result.content.lower() else "en"
    return {"language": lang}


def korean_response(state: State) -> dict:
    response = llm.invoke(
        [
            ("system", "반드시 한국어로만 답변하세요."),
            *state["messages"],
        ]
    )
    return {"messages": [response]}


def english_response(state: State) -> dict:
    response = llm.invoke(
        [
            ("system", "Respond only in English."),
            *state["messages"],
        ]
    )
    return {"messages": [response]}

### 3.2 라우팅 함수

In [ ]:
# ── 라우팅 함수 ──────────────────────────────────────────────────────────────
# add_conditional_edges의 두 번째 인자로 전달되는 함수.
# State를 받아 다음에 실행할 노드 이름(또는 END)을 반환한다.
def route_by_language(state: State) -> Literal["korean_response", "english_response"]:
    return "korean_response" if state.get("language") == "ko" else "english_response"

### 3.3 그래프 조립

보통 엣지(`add_edge`)는 도착 노드가 정해져 있습니다. 조건부 엣지(`add_conditional_edges`)는 출발 노드가 끝난 뒤 라우팅 함수를 실행하고, 그 함수가 돌려준 이름의 노드로 갑니다. 3.2의 `route_by_language`가 `"korean_response"`나 `"english_response"`를 돌려줍니다.

In [ ]:
# ── 그래프 조립 ──────────────────────────────────────────────────────────────
builder = StateGraph(State)
builder.add_node("detect_language", detect_language)
builder.add_node("korean_response", korean_response)
builder.add_node("english_response", english_response)

builder.add_edge(START, "detect_language")

# 조건부 엣지: detect_language 노드 이후 route_by_language 결과에 따라 분기
builder.add_conditional_edges("detect_language", route_by_language)

builder.add_edge("korean_response", END)
builder.add_edge("english_response", END)

app = builder.compile()

### 3.4 질문 넣어 실행하기

**✍️ CHANGE_HERE: 두 번째 질문을 일본어 질문으로 바꾸면 감지 언어와 답은 어떻게 될까요?**

**바꿔 볼 값** — 바로 아래 줄의 `Hello! Tell me about LangGraph.` 자리에 복사해 붙여 넣기

```text
こんにちは！LangGraphについて教えてください。
```

이 셀만 다시 실행합니다. 확인했으면 원래 질문으로 되돌립니다.

In [ ]:
# ── 실행 ─────────────────────────────────────────────────────────────────────
for question in [
    "안녕하세요! LangGraph에 대해 알려주세요.",
    # ✍️ CHANGE_HERE: 두 번째 질문을 일본어 질문으로 바꾸면 감지 언어와 답은 어떻게 될까요?
    "Hello! Tell me about LangGraph.",
]:
    language_graph_result = app.invoke(
        {"messages": [HumanMessage(content=question)], "language": ""}
    )
    results = language_graph_result["messages"]
    for m in results:
        m.pretty_print()
    print(f"감지 언어: {language_graph_result['language']}")
    print()

<details><summary>무슨 일이 생기나</summary>

`감지 언어:`는 `ja`가 아니라 `ko`나 `en` 중 하나로만 나옵니다. 여러 번 실행해 보면 `ko`가 조금 더 자주, `en`도 섞여 나옵니다. 언어를 묻는 프롬프트가 `ko`와 `en` 중에서만 고르게 하고, 라우팅 함수의 갈래도 한국어·영어 두 개뿐이기 때문입니다.

`ko`로 가면 한국어 답변 노드가 답하는데, "한국어로만" 지시가 있어도 일본어로 답할 때가 많습니다. `en`으로 가도 "English only" 지시가 있는데 일본어로 답할 때가 많고, 영어로 답할 때도 있습니다. 일본어를 제대로 나누려면 감지할 값과 답변 노드(갈래)를 함께 늘려야 합니다.

</details>

**볼 지점**
- `감지 언어:` 줄이 라우팅 함수가 본 값입니다. `ko`면 한국어 답변 노드, 아니면 영어 답변 노드로 갔습니다.
- 두 답변 노드는 system 메시지만 다릅니다. 같은 메시지 목록 앞에 "한국어로만" 또는 "English only" 지시를 붙여 LLM을 호출합니다.

## 4. ToolNode로 ReAct 루프 만들기

도구를 가진 Agent를 그래프로 만듭니다. agent 노드가 도구를 쓸지 판단하고, ToolNode가 도구를 실행하고, 다시 agent로 돌아가 결과를 보고 답을 이어 씁니다. 판단 → 도구 실행 → 관찰을 되풀이하는 ReAct 루프입니다. 도구 실행 뒤 LLM을 다시 호출하는 반복을 직접 짜지 않고 `ToolNode`와 `tools_condition`에 맡깁니다.

4.1에서는 환율을, 4.4에서는 질문을 바꿔 다시 실행해 봅니다.

준비 코드입니다. 그래프 도구와 `ToolNode`·`tools_condition`, Agent를 한 번에 만드는 `create_agent`를 불러옵니다. 모델은 4.2에서 도구를 연결하며 준비합니다. 실행만 하면 됩니다.

In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv();

### 4.1 도구 정의

**✍️ CHANGE_HERE: `1350.0`을 `1500.0`으로 바꾸면 4.4의 최종 답은 어떻게 될까요?**

**바꿔 볼 값** — 바로 아래 줄의 `1350.0` 자리에 복사해 붙여 넣기

```text
1500.0
```

이 셀부터 4.4까지 차례로 다시 실행합니다. 확인했으면 `1350.0`으로 되돌리고 이 셀부터 4.4까지 다시 실행합니다.

In [ ]:
@tool
def get_exchange_rate(base_currency: str, target_currency: str) -> float:
    """고정 환율을 반환한다. base_currency와 target_currency는 USD, KRW, EUR 중 하나."""
    rates = {
        # ✍️ CHANGE_HERE: `1350.0`을 `1500.0`으로 바꾸면 4.4의 최종 답은 어떻게 될까요?
        ("USD", "KRW"): 1350.0,
        ("KRW", "USD"): 1 / 1350.0,
        ("EUR", "KRW"): 1470.0,
        ("KRW", "EUR"): 1 / 1470.0,
        ("USD", "EUR"): 0.92,
        ("EUR", "USD"): 1.09,
    }
    if base_currency == target_currency:
        return 1.0
    return rates.get((base_currency, target_currency), -1.0)


@tool
def calculate(expression: str) -> float:
    """간단한 수식을 계산한다. 예: '1350 * 100'"""
    allowed = set("0123456789+-*/(). ")
    if not all(c in allowed for c in expression):
        raise ValueError("허용되지 않는 문자가 포함되어 있습니다.")
    return eval(expression)  # noqa: S307

<details><summary>무슨 일이 생기나</summary>

`get_exchange_rate`의 `Tool Message`에 `1500.0`이 들어오고, 최종 답이 135,000원에서 **150,000원**으로 바뀝니다. `calculate` 도구로 `100 * 1500`까지 계산하는 때가 많습니다.

모델이 금액을 짐작해 답하지 않고, 도구가 돌려준 값을 보고 답을 이어 쓴다는 뜻입니다.

</details>

### 4.2 도구 목록과 LLM 연결

LLM이 도구를 쓰려면 어떤 도구가 있는지 알아야 합니다. `bind_tools()`는 도구의 이름·설명·인자 형식을 LLM에 알려 줍니다. 같은 도구 목록 `tools`를 4.4의 ToolNode도 씁니다.

In [ ]:
tools = [get_exchange_rate, calculate]
llm = get_llm().bind_tools(tools)

### 4.3 agent 노드

`agent_node`는 지금까지의 메시지를 도구가 연결된 `llm`에 넘기고, 받은 AI 메시지를 `messages`에 담아 돌려줍니다. AI 메시지에 도구 호출(`tool_calls`)이 있으면 다음에 ToolNode가 실행되고, 없으면 최종 답으로 끝납니다.

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]


def agent_node(state: State) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

### 4.4 ToolNode로 루프 잇고 실행하기

**✍️ CHANGE_HERE: 질문을 `"안녕! 너는 무엇을 도와줄 수 있어?"`로 바꾸면 출력되는 메시지 칸은 어떻게 달라질까요?**

**바꿔 볼 값** — 바로 아래 줄의 `100달러를 원화로 환산하면 얼마야?` 자리에 복사해 붙여 넣기

```text
안녕! 너는 무엇을 도와줄 수 있어?
```

이 셀만 다시 실행합니다. 확인했으면 원래 질문으로 되돌립니다.

ToolNode는 AI 메시지의 도구 호출을 읽어 해당 도구를 실행하고, 결과를 ToolMessage(도구 결과 메시지)로 돌려주는 미리 만들어진 노드입니다. `tools_condition`은 마지막 메시지에 도구 호출이 있으면 `"tools"` 노드로, 없으면 끝으로 보냅니다. 도구를 실행한 뒤 agent로 돌아가는 엣지가 있어야 모델이 도구 결과를 보고 답을 이어 씁니다. 이 되돌아가는 엣지가 순환 그래프의 고리입니다.

In [ ]:
# ToolNode: AI 메시지에 포함된 tool_calls를 파싱하고 해당 도구를 실행한 뒤
#           결과를 ToolMessage 리스트로 반환한다.
# tools_condition: 마지막 메시지에 tool_calls가 있으면 'tools'로,
#                  없으면(최종 답변이면) END로 라우팅한다.
builder = StateGraph(State)
builder.add_node("agent", agent_node)
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)  # → 'tools' or END
tool_node = ToolNode(tools)
builder.add_node("tools", tool_node)
builder.add_edge("tools", "agent")

app = builder.compile()

print("=== 직접 조립한 ReAct Agent ===")
tool_graph_result = app.invoke(
    # ✍️ CHANGE_HERE: 질문을 `"안녕! 너는 무엇을 도와줄 수 있어?"`로 바꾸면 출력되는 메시지 칸은 어떻게 달라질까요?
    {"messages": [HumanMessage(content="100달러를 원화로 환산하면 얼마야?")]}
)
results = tool_graph_result["messages"]
for m in results:
    m.pretty_print()

print()

<details><summary>무슨 일이 생기나</summary>

출력이 `Human Message`와 `Ai Message` 두 칸으로 끝납니다. `Ai Message`에 `Tool Calls:`가 없고 `Tool Message` 칸도 없습니다. 답에 "계산이나 환율 확인도 도와줄 수 있다"는 말이 나오기도 합니다. 모델은 도구가 있는 것을 알지만, 필요 없는 질문에는 부르지 않습니다.

`tools_condition`이 마지막 메시지에 도구 호출이 없는 것을 보고 tools 노드를 거치지 않고 바로 끝으로 보낸 것입니다.

</details>

### 4.5 create_agent로 한 번에 만들기

In [ ]:
# ── create_agent: 위와 동일한 ReAct 루프를 한 줄로 만드는 편의 함수 ────────────
# 커스텀 노드·엣지가 필요 없을 때 사용. 내부적으로 위와 같은 그래프를 자동 생성한다.
# 멀티에이전트처럼 여러 에이전트가 서로 통신하는 구조는 StateGraph 직접 조립이 필요하다.
print("=== create_agent ===")
agent = create_agent(get_llm(), tools)
agent_result = agent.invoke({"messages": [HumanMessage(content="50유로는 몇 원이야?")]})
results = agent_result["messages"]
for m in results:
    m.pretty_print()

**볼 지점**
- 4.4의 출력에서 `Ai Message` 칸이 두 번 이상 나옵니다. 앞의 것은 도구 호출(`Tool Calls:`)이고, 마지막 것이 최종 답입니다. 도구 결과가 `Tool Message`로 들어온 뒤 agent가 다시 실행된 것입니다.
- 4.5의 `create_agent`는 4.4에서 직접 조립한 agent 노드, tools 노드, 돌아가는 엣지를 한 번에 만들어 줍니다. 노드를 더하거나 흐름을 바꿔야 할 때는 4.4처럼 직접 조립합니다.

## 5. 사람 승인 넣기

메일 보내기 도구를 가진 Agent가 도구를 실행하기 직전에 멈추게 합니다. 사람이 대기 중인 도구 호출을 확인한 뒤 승인하면 이어서 실행하고, 거부하면 거부 결과를 State에 넣고 이어 갑니다.

`send_email` 도구는 `[실제 전송]` 줄을 화면에 출력만 하고 실제 메일은 보내지 않습니다. 이 줄이 보이면 "도구가 실행되었다"는 뜻입니다.

5.3 → 5.4 → 5.5는 5.2에서 만든 그래프와 Checkpointer를 함께 쓰므로 차례로 실행합니다. 처음부터 다시 하려면 5.2부터 실행합니다.

이 장의 준비 코드입니다. 체크포인터(`MemorySaver`)와 도구 결과 메시지(`ToolMessage`)를 함께 불러오고 모델을 준비합니다. 실행을 잊지 않습니다.

In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

### 5.1 도구와 그래프 조립

In [ ]:
@tool
def send_email(recipient: str, subject: str, body: str) -> str:
    """이메일을 전송한다."""
    print(f"\n  [실제 전송] {recipient} / 제목: {subject}")
    return f"이메일이 {recipient}에게 성공적으로 전송되었습니다."


tools = [send_email]
llm_with_tools = llm.bind_tools(tools)
tool_node = ToolNode(tools)


class State(TypedDict):
    messages: Annotated[list, add_messages]


def agent_node(state: State) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


builder = StateGraph(State)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent");

### 5.2 체크포인터와 중단 지점

**✍️ CHANGE_HERE: `interrupt_before=["tools"]`를 `interrupt_before=[]`로 바꾸면 5.3에서 무슨 일이 생길까요?**

**바꿔 볼 값** — 바로 아래 줄의 `["tools"]` 자리에 복사해 붙여 넣기

```text
[]
```

**이 셀과 5.3만** 실행합니다(5.4·5.5는 실행하지 않습니다). 확인했으면 `["tools"]`로 되돌리고 이 셀부터 다시 차례로 실행합니다.

Checkpointer는 실행 상태를 저장합니다. 이 실습에서는 메모리에 저장하는 `MemorySaver`를 씁니다. 저장된 상태가 있어야 멈춘 지점부터 이어 갈 수 있습니다.

`interrupt_before`에 노드 이름을 주면 그 노드를 실행하기 직전에 그래프가 멈춥니다. 메일을 실제로 보내는 `tools` 노드 앞에서 멈춰야 사람이 확인할 수 있습니다. 컴파일할 때 바로 위의 `checkpointer`도 함께 넘깁니다.

In [ ]:
checkpointer = MemorySaver()

# interrupt_before: 지정한 노드 실행 직전에 그래프를 일시 중단한다.
# 사용자가 도구 호출 내용을 확인하고 승인/거부할 수 있다.
# ✍️ CHANGE_HERE: `interrupt_before=["tools"]`를 `interrupt_before=[]`로 바꾸면 5.3에서 무슨 일이 생길까요?
app = builder.compile(checkpointer=checkpointer, interrupt_before=["tools"])

config = {"configurable": {"thread_id": "hitl_demo"}}

<details><summary>무슨 일이 생기나</summary>

5.3에서 멈추지 않고 `[실제 전송] kim@example.com / 제목: 오늘 회의 취소` 줄이 바로 출력됩니다. `대기 중인 도구 호출:` 아래는 비어 있습니다. 그래프가 끝까지 실행되어 마지막 메시지가 도구 호출이 아니라 최종 답이기 때문입니다. 오류는 나지 않습니다. 셀 첫 줄에 "tools 직전에서 중단"이라고 출력되어도, 실제로 멈출지는 `interrupt_before`가 정합니다.

메일 보내기처럼 되돌릴 수 없는 도구는 사람이 확인하기 전에 실행되면 안 됩니다. 그래서 그 도구를 실행하는 노드를 멈출 곳으로 지정합니다.

</details>

### 5.3 1단계: 도구 실행 직전에 멈추기

In [ ]:
# ── 1단계: tools 노드 직전까지 실행 ──────────────────────────────────────────
print("=== 1단계: 에이전트 실행 (tools 직전에서 중단) ===")
app.invoke(
    {
        "messages": [
            HumanMessage(
                content='kim@example.com에게 "오늘 회의 취소" 제목으로 "오후 3시 회의가 취소되었습니다"라는 이메일을 보내줘.'
            )
        ]
    },
    config=config,
)

state = app.get_state(config)
pending_tools = state.values["messages"][-1].tool_calls
print(f"\n대기 중인 도구 호출:")
for tc in pending_tools:
    print(f"  - {tc['name']}({tc['args']})")

### 5.4 2단계: 승인하고 이어 가기

In [ ]:
# ── 2단계: 사용자 승인 후 재개 ────────────────────────────────────────────────
print("\n=== 2단계: 승인 후 재개 ===")
# None을 입력하면 마지막으로 중단된 지점부터 그래프가 이어서 실행된다.
hitl_approve_result = app.invoke(None, config=config)
results = hitl_approve_result["messages"]
for m in results:
    m.pretty_print()

### 5.5 거부하고 이어 가기

**✍️ CHANGE_HERE: `"hitl_reject"`를 5.2와 같은 `"hitl_demo"`로 바꾸면 출력되는 메시지는 어떻게 달라질까요?**

**바꿔 볼 값** — 바로 아래 줄의 `hitl_reject` 자리에 복사해 붙여 넣기

```text
hitl_demo
```

5.2부터 이 셀까지 차례로 다시 실행합니다. 확인했으면 `hitl_reject`로 되돌립니다.

In [ ]:
# ── 거부 시나리오 ─────────────────────────────────────────────────────────────
print("\n=== 거부 시나리오 ===")
# ✍️ CHANGE_HERE: `"hitl_reject"`를 5.2와 같은 `"hitl_demo"`로 바꾸면 출력되는 메시지는 어떻게 달라질까요?
config2 = {"configurable": {"thread_id": "hitl_reject"}}
app.invoke(
    {
        "messages": [
            HumanMessage(
                content='boss@example.com에게 "긴급" 제목으로 "즉시 연락 바랍니다"라는 이메일을 보내줘.'
            )
        ]
    },
    config=config2,
)

# 거부: tool_calls를 직접 ToolMessage로 처리해 상태에 삽입한 뒤 재개
state2 = app.get_state(config2)
rejected_calls = state2.values["messages"][-1].tool_calls

rejection_messages = [
    ToolMessage(content="사용자가 이메일 전송을 취소했습니다.", tool_call_id=tc["id"])
    for tc in rejected_calls
]
app.update_state(config2, {"messages": rejection_messages}, as_node="tools")
hitl_reject_result = app.invoke(None, config=config2)
results = hitl_reject_result["messages"]
for m in results:
    m.pretty_print()

<details><summary>무슨 일이 생기나</summary>

5.5의 출력이 4칸에서 8칸으로 늘어납니다. 앞의 4칸은 5.3·5.4의 kim@example.com 대화(질문, 도구 호출, 전송 결과, 답)이고, 그 뒤에 boss@example.com 대화가 이어집니다. boss@example.com 메일은 여전히 보내지 않습니다(`[실제 전송] boss@example.com` 줄이 없습니다).

Checkpointer는 `thread_id`마다 대화를 따로 저장합니다. 같은 번호를 주면 앞 대화에 이어 붙고, 다른 번호를 주면 새 대화로 시작합니다. 사용자나 세션마다 대화를 나눌 때 이 번호를 다르게 줍니다.

</details>

**볼 지점**
- 승인 흐름에서는 1단계 뒤에 멈췄다가 `app.invoke(None, ...)`로 멈춘 곳부터 이어 갔습니다.
- 거부 흐름에서는 `app.update_state(...)`가 거부 메시지를 도구 결과처럼 State에 넣은 뒤 이어 갔습니다. 그래서 `send_email`이 실행되지 않았습니다.

## 6. 대화형 승인 Agent

5장의 승인·거부를 사람이 직접 고르는 프로그램으로 만듭니다. Agent가 도구를 쓰려고 할 때마다 멈추고, 입력창에서 `y`(승인), `n`(거부), `q`(종료) 중 하나를 고릅니다.

코드는 완성되어 있습니다. 6.1~6.3을 실행해 그래프와 함수를 준비하고, 6.4를 실행해 직접 요청해 봅니다. 6.4를 실행하면 입력창이 열립니다. 요청에는 받는 사람·제목·본문을 모두 적습니다. 아래 문장을 그대로 붙여 넣어도 됩니다.

`lee@example.com에게 "점심 약속" 제목으로 "12시에 1층 로비에서 만나요"라는 이메일을 보내줘.`

이어서 `y`/`n`/`q`를 입력합니다. `n`을 고르면 거부 사유도 묻습니다. 예를 들어 `n`을 고르고 사유에 `제목을 "점심 약속 변경"으로 바꿔서 다시 제안해`를 적은 뒤, 에이전트가 사유를 보고 어떻게 답하는지 봅니다. 끝나지 않으면 `q`를 입력하거나 실행을 중지(■)합니다.

`q`나 ■로 멈춘 뒤 6.4를 다시 실행하면 오류가 날 수 있습니다. 멈춘 대화가 그대로 남아 있기 때문입니다. 이때는 6.1부터 다시 실행합니다.

준비 코드는 5장과 같은 도구를 불러오고 모델을 준비합니다. 펼치지 않고 실행만 합니다.

In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

### 6.1 도구와 그래프 조립

In [ ]:
@tool
def send_email(recipient: str, subject: str, body: str) -> str:
    """이메일을 전송한다."""
    print(f"\n  [실제 전송] {recipient} / 제목: {subject}")
    return f"이메일이 {recipient}에게 성공적으로 전송되었습니다."


tools = [send_email]
llm_with_tools = llm.bind_tools(tools)
tool_node = ToolNode(tools)


class State(TypedDict):
    messages: Annotated[list, add_messages]


def agent_node(state: State) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


builder = StateGraph(State)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

app = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["tools"],
)

### 6.2 승인·거부를 묻는 함수

In [ ]:
def print_tool_calls(tool_calls: list[dict]) -> None:
    print("\n실행 대기 중인 도구 호출")
    for index, tool_call in enumerate(tool_calls, start=1):
        print(f"{index}. {tool_call['name']}")
        for key, value in tool_call["args"].items():
            print(f"   - {key}: {value}")


def reject_tool_calls(config: dict, tool_calls: list[dict], reason: str) -> None:
    rejection_messages = [
        ToolMessage(
            content=f"사용자가 도구 실행을 거부했습니다. 사유: {reason}",
            tool_call_id=tool_call["id"],
        )
        for tool_call in tool_calls
    ]
    app.update_state(config, {"messages": rejection_messages}, as_node="tools")


def ask_human_decision() -> tuple[str, str]:
    print("\n도구를 실행할까요?")
    print("- y: 승인 후 실행")
    print("- n: 거부 후 에이전트에게 취소 사유 전달")
    print("- q: 프로그램 종료")

    decision = input("선택(y/n/q): ").strip().lower()
    if decision == "n":
        reason = input("거부 사유를 입력하세요: ").strip()
        return decision, reason
    return decision, ""

### 6.3 Agent 실행 루프

In [ ]:
def run_agent(user_input: str, thread_id: str = "hitl_demo") -> None:
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke({"messages": [HumanMessage(content=user_input)]}, config=config)

    while True:
        state = app.get_state(config)
        tool_calls = state.values["messages"][-1].tool_calls

        if not tool_calls:
            print("\n에이전트 응답")
            print(result["messages"][-1].content)
            break

        print_tool_calls(tool_calls)
        decision, reason = ask_human_decision()

        if decision == "y":
            result = app.invoke(None, config=config)
        elif decision == "n":
            reject_tool_calls(config, tool_calls, reason or "사유 없음")
            result = app.invoke(None, config=config)
        elif decision == "q":
            print("프로그램을 종료합니다.")
            break
        else:
            print("y, n, q 중 하나를 입력하세요.")

### 6.4 직접 실행하기

In [ ]:
if __name__ == "__main__":
    request = input("에이전트에게 요청할 내용을 입력하세요: ")
    run_agent(request)

**볼 지점**
- `run_agent`는 멈출 때마다 State의 마지막 메시지에서 도구 호출을 꺼내 보여 줍니다. 사람이 고른 값에 따라 이어 가거나(`y`), 거부 메시지를 넣고 이어 가거나(`n`), 끝냅니다(`q`).
- `n`을 고르면 `reject_tool_calls`가 5장의 거부와 같은 방식(`app.update_state`)으로 거부 사유를 State에 기록합니다.

## 7. 정리

LangGraph 그래프를 다섯 단계로 실행해 보았습니다.

| 장 | 만든 것 | 핵심 코드 |
|---|---|---|
| 2. 기본 그래프 | START → chatbot → END | `add_node`, `add_edge`, `compile` |
| 3. 조건부 엣지 | 언어에 따라 한국어·영어 노드로 분기 | `add_conditional_edges` |
| 4. ToolNode | 도구 호출 → 도구 실행 → 다시 판단하는 루프 | `ToolNode`, `tools_condition` |
| 5. 사람 승인 | 도구 실행 직전에 멈추고 승인·거부 | `MemorySaver`, `interrupt_before`, `update_state` |
| 6. 대화형 승인 | 사람이 y/n/q로 고르는 실행 루프 | `get_state`, `input()` |

그래프는 노드를 더하고 엣지로 이어 조립합니다. 반복은 되돌아가는 엣지로, 분기는 조건부 엣지로, 사람 승인은 Checkpointer와 중단 지점으로 만듭니다.

값을 바꿔 다시 실행하며 Reducer, 조건부 엣지의 갈래, 도구 호출 여부, 멈출 노드, 대화 번호(`thread_id`)가 그래프의 동작을 어떻게 바꾸는지 확인했습니다.